# PS5 final model training and export

This notebook is for local reproduction. The competition package in **PS5-Group2** runs the fitted model arrays directly.

- Fraud uses a 26-feature random forest and XGBoost trained on the supplied fraud training CSV. The blend and the threshold are frozen.
- Activity uses the original four frozen activity fold models, with a class-frequency correction and a refit class-balanced RBF SVM in each fold. A clean refit of the original activity boosters previously diverged from those frozen fold models even with pinned packages, so the notebook uses the archived fold models as the reproducible starting point. Their original training code is included below for inspection.
- The notebook exports both NPZ models to a new directory and compares every array against the files in **PS5-Group2**. It stops with a parameter diff if an exact match cannot be achieved. It reads only the original training data and the saved base models.

Use a Python 3.12 kernel with the pinned modeling libraries listed below. For the original fraud fit, use a CUDA-capable XGBoost build and GPU. On the build desktop the existing project interpreter is in the local .venv-overnight environment. The course Python environment is for inference.

In [ ]:
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
from types import SimpleNamespace
from zipfile import ZipFile
import json
import pickle
import sys
from tempfile import mkdtemp

import numpy as np
import pandas as pd
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from lightgbm import LGBMClassifier
from scipy.special import logit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "PS5-Group2").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from this repository.")
REFERENCE = ROOT / "PS5-Group2" / "best_models"
DRAFT = ROOT / "Code Prototypes" / "PS5 Team Eval Draft"
FRAUD_ARCHIVE = ROOT / "Docs and Data" / "610_ps4_training" / "trainingT1FD" / "cct_train.zip"
FRAUD_THRESHOLD = 0.6603664809465408
SEED = 610
WORKERS = 8
PINS = {
    "numpy": "2.5.5", "pandas": "3.0.1", "scipy": "1.18.1", "scikit-learn": "1.9.1",
    "imbalanced-learn": "0.14.2", "xgboost": "3.4.1", "lightgbm": "4.7.0",
    "cloudpickle": "3.1.3", "threadpoolctl": "3.8.0", "joblib": "1.6.0",
}
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the Python 3.12 training kernel.")
differences = {name: (version(name), want) for name, want in PINS.items() if version(name) != want}
if differences:
    raise RuntimeError(f"Training libraries differ from the frozen run: {differences}")
build = ROOT / ".local-ps5-experiments" / "course-build"
build.mkdir(parents=True, exist_ok=True)
(build.parent / ".gitignore").write_text("*\n", encoding="utf-8") if not (build.parent / ".gitignore").exists() else None
MODELS = Path(mkdtemp(prefix="system-exports-", dir=build))
print("Reproduction outputs:", MODELS)

## Fraud features and final estimators

Transactions are sorted by their Unix timestamp before computing strictly earlier history. The final forest and booster share the same median imputation and missing-value indicators. The threshold is the value recorded at freeze, so this notebook performs no model selection.

In [ ]:
def prior_counts(times, days):
    """Count earlier events in a trailing window, excluding simultaneous events."""
    return times.searchsorted(times) - times.searchsorted(times - days * 86400)


def fraud_features(events):
    # Return features in input order, including when called on a shuffled batch.
    d = events.reset_index(drop=True).sort_values("unix_time", kind="stable")
    x = pd.DataFrame(index=d.index)
    x["amount"] = d.amt
    x["hour"] = d.trans_time.str.slice(0, 2).astype(int)
    lat1, lat2 = np.radians(d.lat), np.radians(d.merch_lat)
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(np.radians(d.merch_long-d.long)/2)**2
    x["home_merchant_km"] = 12742 * np.arcsin(np.sqrt(a.clip(0, 1)))

    d["time_band"] = x["hour"] // 6
    entities = {"card":["cc_num"], "card_category":["cc_num","category"],
                "account":["acct_num"], "person":["first","last","dob"],
                "card_merchant":["cc_num","merchant"], "category":["category"], "merchant":["merchant"],
                "merchant_time":["merchant","time_band"], "card_time":["cc_num","time_band"]}
    for prefix, keys in entities.items():
        # Collapse simultaneous events so their within-second ordering is immaterial.
        group_keys = keys + ["unix_time"]
        present = d[keys].notna().all(axis=1)
        batch = (d.loc[present].groupby(group_keys, sort=False, observed=True)
                 .amt.agg(total="sum", n="size").reset_index())
        groups = batch.groupby(keys, sort=False, observed=True)
        count = groups.n.cumsum() - batch.n
        total = groups.total.cumsum() - batch.total
        batch["count"] = count
        batch["mean"] = total / count.replace(0, np.nan)
        batch["elapsed"] = groups.unix_time.diff()
        prior = d[group_keys].merge(batch[group_keys + ["count","mean","elapsed"]],
                                    on=group_keys, how="left", sort=False, validate="many_to_one")
        prior.index = d.index
        x[prefix+"_count"] = prior["count"].fillna(0)
        x[prefix+"_amount_ratio"] = d.amt / prior["mean"].replace(0, np.nan)
        if prefix == "card":
            x["card_elapsed"] = prior["elapsed"]
    # Prefix sums and binary search make strict trailing windows inexpensive per card.
    for name in ("hour_count","day_count","day_spend"):
        x[name] = 0.
    x["reference_days_observed"] = 0.
    x["day_vs_prior28_ratio"] = np.nan
    for _, block in d.groupby("cc_num", sort=False):
        times = block.unix_time.to_numpy()
        end = times.searchsorted(times, side="left")
        hour = times.searchsorted(times-3600)
        day = times.searchsorted(times-86400)
        sums = np.r_[0., block.amt.to_numpy().cumsum()]
        x.loc[block.index, ["hour_count","day_count","day_spend"]] = np.column_stack(
            [end-hour, end-day, sums[end]-sums[day]])
        ref_start = times.searchsorted(times-29*86400, side="left")
        ref_total = sums[day]-sums[ref_start]
        coverage = np.clip((times-times[0])/86400-1, 0, 28)
        ratio = np.full(len(block), np.nan)
        np.divide(sums[end]-sums[day], ref_total/28, out=ratio,
                  where=(coverage == 28) & (ref_total > 0))
        x.loc[block.index, ["reference_days_observed","day_vs_prior28_ratio"]] = np.column_stack([coverage,ratio])
    # Count category and overall traffic over aligned historical windows.
    t = d.unix_time.to_numpy()
    c7=np.zeros(len(t),int); c35=c7.copy()
    for idx in d.reset_index(drop=True).groupby("category",sort=False).indices.values():
        c7[idx]=prior_counts(t[idx],7); c35[idx]=prior_counts(t[idx],35)
    n7,n35=prior_counts(t,7),prior_counts(t,35)
    x["recent_category_share"]=(c7+.5)/(n7+7)
    x["category_share_change"]=np.log(x.recent_category_share.to_numpy()/((c35-c7+.5)/(n35-n7+7)))
    return x.sort_index().astype("float32")

In [ ]:
ID_FIELDS = ["cc_num", "acct_num", "first", "last", "dob", "merchant", "category"]
EVENT_FIELDS = ID_FIELDS + ["unix_time", "trans_time", "trans_date", "amt", "lat", "long", "merch_lat", "merch_long"]
FRAUD_COLUMNS = [
    "amount", "hour", "card_amount_ratio", "card_category_amount_ratio", "hour_count", "day_count",
    "card_elapsed", "day_spend", "card_count", "card_merchant_count", "home_merchant_km",
    "account_count", "account_amount_ratio", "person_count", "person_amount_ratio",
    "merchant_count", "merchant_amount_ratio", "reference_days_observed", "day_vs_prior28_ratio",
    "card_merchant_amount_ratio", "card_category_count", "merchant_time_count", "merchant_time_amount_ratio",
    "card_time_amount_ratio", "recent_category_share", "category_share_change",
]
with ZipFile(FRAUD_ARCHIVE) as z:
    member, = [n for n in z.namelist() if Path(n).name == "cct_train.csv"]
    with z.open(member) as source:
        fraud = pd.read_csv(source, usecols=EVENT_FIELDS + ["is_fraud"], dtype=dict.fromkeys(ID_FIELDS, "string"))
fraud = fraud.sort_values("unix_time", kind="stable").reset_index(drop=True)
fraud_y = fraud.pop("is_fraud")
fraud_x = fraud_features(fraud)[FRAUD_COLUMNS].to_numpy(dtype=np.float32)

rf = make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                   RandomForestClassifier(n_estimators=100, max_samples=0.5, min_samples_leaf=2,
                                          random_state=SEED, n_jobs=WORKERS))
rf.fit(fraud_x, fraud_y)
rf[-1].n_jobs = 1

gb = make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                   XGBClassifier(n_estimators=240, max_leaves=16, min_child_weight=1.6, learning_rate=0.05,
                                 subsample=0.9632086833875929, colsample_bylevel=1.0, colsample_bytree=1.0,
                                 reg_alpha=0.00417650411810458, reg_lambda=0.6036774229389704,
                                 max_depth=0, grow_policy="lossguide", tree_method="hist", random_state=10242048,
                                 enable_categorical=True, device="cuda", n_jobs=2))
gb.fit(fraud_x, fraud_y)
device = json.loads(gb[-1].get_booster().save_config())["learner"]["generic_param"]["device"]
if not device.startswith("cuda"):
    raise RuntimeError(f"Expected CUDA training for exact reproduction, got {device}.")
gb[-1].set_params(device="cpu", n_jobs=1)
fraud_model = SimpleNamespace(fields=EVENT_FIELDS, columns=FRAUD_COLUMNS, threshold=FRAUD_THRESHOLD,
                              forest=SimpleNamespace(members=[(0.75, rf), (0.25, gb)]))
print(f"Fitted fraud components on {len(fraud_y):,} purchases.")

## Activity base recipe

These definitions record the original four-fold training recipe. The final build loads the saved outputs of this recipe and fits the new SVM heads in those same folds. The saved base is part of the input for exact reproduction.

In [ ]:
def activity_booster(fold):
    """Recreate the previously selected native booster configuration for this fold."""
    if fold == 1:
        return LGBMClassifier(n_estimators=100, num_leaves=4, min_child_samples=16,
                              learning_rate=0.07832503615697724, max_bin=511,
                              colsample_bytree=0.8814460015050225, reg_alpha=0.0009765625,
                              reg_lambda=0.024037654653852953, n_jobs=WORKERS, verbose=-1)
    return XGBClassifier(n_estimators=56, max_leaves=4, max_depth=0, grow_policy="lossguide",
                         tree_method="hist", enable_categorical=True, verbosity=0, random_state=10242048,
                         min_child_weight=2.707170950755067, learning_rate=0.4810290992611119,
                         subsample=0.9659488304727971, colsample_bytree=0.9692046632582698,
                         reg_alpha=0.0009765625, reg_lambda=0.1384884162470523, n_jobs=WORKERS)


def group_folds(x, y, groups, count, class_count):
    """Repeat the original deterministic split search to keep all training classes."""
    for seed in range(SEED, SEED + 200):
        folds = list(StratifiedGroupKFold(count, shuffle=True, random_state=seed).split(x, y, groups))
        if all(len(np.unique(y[a])) == class_count and
               (count == 3 or len(np.unique(y[b])) == class_count) for a, b in folds):
            return folds
    raise ValueError("No grouped split covered all required classes.")


def neighbor_shares(ref, query, ids, groups, class_count):
    """One vote per distinct reading; conflicting labels split that reading's vote."""
    scale = StandardScaler().fit(ref)
    keep = scale.var_ > 0
    train, target = scale.transform(ref)[:, keep], scale.transform(query)[:, keep]
    unique, first, inverse = np.unique(groups, return_index=True, return_inverse=True)
    votes = np.zeros((len(unique), class_count))
    np.add.at(votes, (inverse, ids), 1)
    votes /= votes.sum(axis=1, keepdims=True)
    distance, near = NearestNeighbors(n_neighbors=15).fit(train[first]).kneighbors(target)
    weights = 1 / np.maximum(distance, 1e-8)
    return np.sum(votes[near] * weights[:, :, None], axis=1) / weights.sum(axis=1, keepdims=True)


def fit_activity_fold(x, y, groups, labels, fold):
    """Fit preprocessing, cross-fitted neighbor features and the three component boosters."""
    count = len(labels)
    pair = np.array([np.flatnonzero(labels == sport)[0] for sport in ("Football", "Tennis")])
    scale = StandardScaler().fit(x)
    sensors = scale.transform(x)
    meta = np.empty((len(x), count))
    for train, valid in group_folds(x, y, groups, 3, count):
        meta[valid] = neighbor_shares(x[train], x[valid], y[train], groups[train], count)
    share_scale = StandardScaler().fit(meta)
    augmented = np.column_stack((sensors, share_scale.transform(meta)))
    general = activity_booster(fold).fit(augmented, y, sample_weight=compute_sample_weight("balanced", y))
    sports = np.isin(y, pair)
    specialist = activity_booster(fold).fit(augmented[sports], (y[sports] == pair[1]).astype(int))

    # Lift classes below median support, then remove Tomek links within this training fold.
    counts = np.bincount(y)
    target = int(np.median(counts))
    strategy = {i: target for i, n in enumerate(counts) if n < target}
    sampler = SMOTETomek(smote=SMOTE(sampling_strategy=strategy, k_neighbors=min(3, counts.min()-1),
                                      random_state=SEED),
                         tomek=TomekLinks(sampling_strategy="all", n_jobs=1), random_state=SEED)
    xr, yr = sampler.fit_resample(sensors, y)
    resampled = activity_booster(fold).fit(xr, yr)
    return dict(x=x, y=y, groups=groups, count=count, pair=pair, scale=scale, share_scale=share_scale,
                general=general, specialist=specialist, resampled=resampled)

In [ ]:
activity = pd.read_csv(DRAFT / "har_train.csv", header=None)
raw = activity.iloc[:, :-1]
x = raw.to_numpy()
labels, y = np.unique(activity.iloc[:, -1], return_inverse=True)
groups = pd.util.hash_pandas_object(raw, index=False).to_numpy()
base_path = DRAFT / "best_models" / "activity_best_15_neighbor_sports_booster_ensemble.pkl"
base_bytes = base_path.read_bytes()
expected = "cd84b41ce7001278ea204bf6cbfd5e5de2a7424af5ac8fb414c3acbd95dcc8b7"
if sha256(base_bytes).hexdigest() != expected:
    raise ValueError("The archived activity base model differs from the one used in the final fit.")
# This trusted local pickle supplies the previously fitted base folds.
archived = pickle.loads(base_bytes)
np.testing.assert_array_equal(archived.labels, labels)

folds = StratifiedGroupKFold(4, shuffle=True, random_state=SEED).split(x, y, groups)
side = []
with threadpool_limits(limits=1, user_api="blas"):
    for fold, (train, valid) in enumerate(folds):
        base = archived.models[fold]
        assert not np.intersect1d(groups[train], groups[valid]).size
        np.testing.assert_array_equal(base["x"], x[train])
        np.testing.assert_array_equal(base["groups"], groups[train])
        np.testing.assert_array_equal(base["y"], y[train])
        svm = make_pipeline(StandardScaler(), SVC(C=4, class_weight="balanced", probability=True, random_state=SEED))
        side.append(svm.fit(x[train], y[train]))

activity_model = SimpleNamespace(main=SimpleNamespace(models=archived.models, labels=labels, prior=0.075, mix=0),
                                 side=side, weight=0.35)
print(f"Fitted {len(side)} activity SVMs on the original grouped training folds.")

## Export the fitted arrays and compare with the submission

The export uses the same tree representation, class order, neighbor reference, and SVM probability parameters as the submission evaluator. All comparisons read the submitted arrays as references. Only the local reproduction directory receives output files.

In [ ]:
def write_trees(out, prefix, trees, classes, base, objective, kind):
    nodes = {key: [] for key in ("feature", "left", "right", "cutoff", "value", "missing_left")}
    roots = []
    for tree in trees:
        start = len(nodes["left"])
        roots.append(start)
        for key in nodes:
            values = np.asarray(tree[key])
            if key in ("left", "right"):
                values = np.where(values < 0, -1, values + start)
            nodes[key].extend(values)
    for key, values in nodes.items():
        dtype = (np.bool_ if key == "missing_left" else np.int32 if key in ("feature", "left", "right")
                 else np.float32 if kind == "xgb" else np.float64)
        out[prefix + key] = np.asarray(values, dtype=dtype)
    out[prefix + "root"] = np.asarray(roots, dtype=np.int32)
    out[prefix + "class"] = np.asarray(classes, dtype=np.int32)
    out[prefix + "base"] = np.asarray(base, dtype=np.float32 if kind == "xgb" else np.float64)
    out[prefix + "objective"] = np.array(objective)
    out[prefix + "kind"] = np.array(kind)


def forest(out, prefix, model):
    trees = []
    assert np.array_equal(model.classes_, [0, 1])
    for estimator in model.estimators_:
        tree = estimator.tree_
        trees.append(dict(feature=tree.feature, left=tree.children_left, right=tree.children_right,
                          cutoff=tree.threshold, value=tree.value[:, 0, 1],
                          missing_left=tree.missing_go_to_left))
    write_trees(out, prefix, trees, [0] * len(trees), [0.0], "mean", "forest")


def booster(out, prefix, model):
    if hasattr(model, "get_booster"):
        learned = json.loads(model.get_booster().save_raw(raw_format="json"))["learner"]
        model_data = learned["gradient_booster"]["model"]
        trees = []
        for tree in model_data["trees"]:
            assert not any(tree["split_type"])
            trees.append(dict(feature=tree["split_indices"], left=tree["left_children"],
                              right=tree["right_children"], cutoff=tree["split_conditions"],
                              value=tree["split_conditions"], missing_left=tree["default_left"]))
        name = learned["objective"]["name"]
        base = np.atleast_1d(json.loads(learned["learner_model_param"]["base_score"]))
        assert name in ("binary:logistic", "multi:softprob")
        if name == "binary:logistic":
            base = logit(base)
        write_trees(out, prefix, trees, model_data["tree_info"], base,
                    "logistic" if name.startswith("binary:") else "softmax", "xgb")
        return

    def flatten(node, collected):
        index = len(collected)
        collected.append(None)
        if "leaf_value" in node:
            collected[index] = (0, -1, -1, 0.0, node["leaf_value"], True)
        else:
            assert node["decision_type"] == "<=" and node["missing_type"] != "Zero"
            left = flatten(node["left_child"], collected)
            right = flatten(node["right_child"], collected)
            collected[index] = (node["split_feature"], left, right, node["threshold"], 0.0, node["default_left"])
        return index

    dump = model.booster_.dump_model()
    count = dump["num_tree_per_iteration"]
    assert dump["objective"] == "binary sigmoid:1" or dump["objective"].startswith("multiclass ")
    trees = []
    for tree in dump["tree_info"]:
        rows = []
        flatten(tree["tree_structure"], rows)
        trees.append(dict(zip(("feature", "left", "right", "cutoff", "value", "missing_left"), zip(*rows))))
    write_trees(out, prefix, trees, np.arange(len(trees)) % count, np.zeros(count),
                "logistic" if count == 1 else "softmax", "lgbm")


def export_fraud(predictor):
    (forest_weight, pipeline), (boost_weight, other) = predictor.forest.members
    imp = pipeline[0]
    np.testing.assert_array_equal(imp.statistics_, other[0].statistics_)
    np.testing.assert_array_equal(imp.indicator_.features_, other[0].indicator_.features_)
    assert np.isfinite(imp.statistics_).all()
    data = dict(fields=predictor.fields, columns=predictor.columns, threshold=predictor.threshold,
                fill=imp.statistics_.astype(np.float32), indicator=imp.indicator_.features_,
                forest_weight=forest_weight, boost_weight=boost_weight)
    forest(data, "rf_", pipeline[-1])
    booster(data, "gb_", other[-1])
    np.savez_compressed(MODELS / "fraud.npz", **data)


def export_activity(predictor):
    base = predictor.main
    assert base.mix == 0
    data = dict(labels=base.labels.astype(str), svm_weight=predictor.weight,
                prior_exponent=base.prior, fold_count=len(base.models))
    for i, (m, side) in enumerate(zip(base.models, predictor.side)):
        prefix = f"fold{i}_"
        ref, y = m["x"], m["y"]
        scaler = StandardScaler().fit(ref)
        _, first, inverse = np.unique(m["groups"], return_index=True, return_inverse=True)
        votes = np.zeros((len(first), m["count"]))
        np.add.at(votes, (inverse, y), 1)
        votes /= votes.sum(axis=1, keepdims=True)
        keep = scaler.var_ > 0
        for key, value in dict(refs=scaler.transform(ref)[first][:, keep], varying=keep, votes=votes,
                               ref_mean=scaler.mean_, ref_scale=scaler.scale_, pair=m["pair"],
                               frequencies=np.bincount(y) / len(y),
                               sensor_mean=m["scale"].mean_, sensor_scale=m["scale"].scale_,
                               share_mean=m["share_scale"].mean_, share_scale=m["share_scale"].scale_,
                               svm_mean=side[0].mean_, svm_scale=side[0].scale_).items():
            data[prefix + key] = value
        svc = side[-1]
        np.testing.assert_array_equal(svc.classes_, np.arange(len(base.labels)))
        mapping = dict(support="support_", support_vectors="support_vectors_", n_support="_n_support",
                       dual="_dual_coef_", intercept="_intercept_", prob_a="_probA", prob_b="_probB", gamma="_gamma")
        for key, attr in mapping.items():
            data[prefix + key] = getattr(svc, attr)
        for component in ("general", "specialist", "resampled"):
            booster(data, prefix + component + "_", m[component])
    np.savez_compressed(MODELS / "activity.npz", **data)

In [ ]:
export_fraud(fraud_model)
export_activity(activity_model)
mismatches = {}
for name in ("fraud.npz", "activity.npz"):
    path = MODELS / name
    with np.load(REFERENCE / name, allow_pickle=False) as expected, np.load(path, allow_pickle=False) as rebuilt:
        different = sorted(set(expected.files) ^ set(rebuilt.files))
        different.extend(key for key in expected.files if key in rebuilt.files and
                         (expected[key].dtype != rebuilt[key].dtype or not np.array_equal(expected[key], rebuilt[key])))
    if different:
        mismatches[name] = sorted(different)
    else:
        identical_file = sha256(path.read_bytes()).digest() == sha256((REFERENCE / name).read_bytes()).digest()
        print(f"{name}: all exported arrays match; identical NPZ bytes: {identical_file}")
if mismatches:
    raise AssertionError(f"Rebuilt parameters differ from the frozen exports: {mismatches}")